In [1]:
# =============================================================================
# Antibiotic Resistance Prediction using Proximity, Frequency, and G-score
# =============================================================================
# 
# This notebook trains and evaluates Random Forest classifiers on mutation-level
# features (mutation frequency, 3D structural proximity to known resistance-conferring
# mutations, and g-scores) to predict antibiotic resistance in Mycobacterium tuberculosis.
# It includes baseline modeling,combined modeling and predictions for uncertain mutations (Category 3).

In [ ]:
## load dependencies
import pandas as pd
import numpy as np
import re
from evcouplings.compare import DistanceMap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score,roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from scipy import stats
import os,json

In [ ]:
from utils import *

## data preprocessing

In [ ]:
# ----------------------------
# STEP 1: Load and Inspect Dataset
# ----------------------------
final_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")
print("Unique genes:", np.unique(final_df['gene']))
print("Data shape:", final_df.shape)

In [ ]:
# STEP 2: Basic Preprocessing
# ----------------------------
# Separate Category 3 entries (uncertain mutations)
category_3_data = final_df[final_df['confidence'] == "3) Uncertain significance"]

# Retain only labeled data (non-Category 3) for supervised learning
label_data = final_df[final_df['confidence'] != "3) Uncertain significance"]

# Target conversion to binary
label_data['binary_confidence'] = label_data['confidence'].map({
    "1) Assoc w R": 0,
    "2) Assoc w R - Interim": 0,
    "4) Not assoc w R - Interim": 1,
    "5) Not assoc w R": 1
})

# Drop missing targets
label_data = label_data[label_data['binary_confidence'].notna()]

#some debugging steps
print(label_data['confidence'].value_counts())
print(np.unique(label_data['confidence']))
# Identify rows where the 'frequency' column is NaN
nan_frequency_rows = label_data[label_data['frequency'].isna()]

diverse_sample_mutation_table = label_data.groupby('frequency').first().reset_index()
diverse_sample_mutation_table.head()

## run the baseline and combined models

In [ ]:
feature_sets = {
    "Frequency": ['frequency'],
    "Proximity": ['Proximity_to_R_Conferring'],
    "Proximity_1d":["Proximity_1D"],
    "G-score": ['G_score'],
    "Frequency + Proximity": ['frequency', 'Proximity_to_R_Conferring'],
    "Frequency + G-score": ['frequency', 'G_score'],
    "Combined": ['frequency', 'Proximity_to_R_Conferring', 'G_score']
}


In [ ]:
results = {}

for label, numeric_columns in feature_sets.items():
    print(f"\n=== Running model for: {label} ===")

    # Step 3: Feature selection
    label_data[numeric_columns] = label_data[numeric_columns].fillna(0)
    
    # Step 4: Split
    X_train, X_test, y_train, y_test, train_indices, test_indices = data_split(label_data, numeric_columns)
    
    # Step 5: Train and evaluate
    trained_rf, y_pred_test, y_prob_test = model_training(X_train, y_train, X_test)
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_test, labels=[0,1]) #0-resistant
    print("Confusion matrix", cm)
    tn,fp,fn, tp = cm.ravel()
    sensitivity = tp / (tp+fn) #sensitivity resistant
    specificity = tn / (tn+fp) # specificity resistant
    print(f" Sensitivity: {sensitivity}, specificity: {specificity}")
    accuracy, roc_auc, precision = performance_evaluation(y_test, y_pred_test, y_prob_test)
    print("Accuracy:", accuracy)
    print("AUC:", roc_auc)
    print("Precision:", precision)

    # Step 6: Feature importance
    # feature_importances = plot_feature_importance(trained_rf, numeric_columns)
    feature_importances = trained_rf.feature_importances_

    # Step 7: Category 3 prediction
    (
        misclassified_original_indices,
        misclassified_samples,
        confidence_0_count,
        confidence_0_percentage,
        confidence_1_count,
        confidence_1_percentage
    ) = predict_category_3(
        category_3_data,
        trained_rf,
        final_df,
        numeric_columns,
        y_test,
        y_pred_test,
        test_indices
    )

    # Store all results
    results[label] = {
        "features": numeric_columns,
        "accuracy": accuracy,
        "precision": precision,
        "roc_auc": roc_auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "feature_importance": feature_importances,
        "misclassified_indices": misclassified_original_indices,
        "misclassified_samples": misclassified_samples,
        "category_3_predictions": {
            "resistant_count": confidence_0_count,
            "susceptible_count": confidence_1_count,
            "resistant_percentage": confidence_0_percentage,
            "susceptible_percentage": confidence_1_percentage
        }
    }


In [ ]:
def find_invalid_keys(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if not isinstance(k, (str, int, float, bool, type(None))):
                print(f"[Invalid Key] Path: {path} → Key: {k} (type: {type(k).__name__})")
            find_invalid_keys(v, path=f"{path}['{k}']")
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            find_invalid_keys(item, path=f"{path}[{i}]")


# Check before saving
find_invalid_keys(results)

def convert_keys_to_str(obj):
    if isinstance(obj, dict):
        return {str(k): convert_keys_to_str(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_keys_to_str(i) for i in obj]
    else:
        return obj
cleaned_results = convert_keys_to_str(results)

In [20]:
with open("../results/baseline_combined_results.json", "w") as f:
    for label in cleaned_results:
        for key in cleaned_results[label]:
            if not isinstance(key, (str, int, float, bool, type(None))):
                print("Invalid key type:", key, type(key))
    json.dump(cleaned_results, f, indent=4, default=str)

print("\nAll model combinations evaluated and saved.")


All model combinations evaluated and saved.


In [21]:
def generate_markdown_table(results_dict):
    table_md = "| Model | Accuracy | AUC (avg) | Precision | Sensitivity | Specificity\n"
    table_md += "|--------|----------|------------|-----------|-----------|-----------|\n"

    for model_name, metrics in results_dict.items():
        acc = f"{metrics.get('accuracy', 0) * 100:.1f}%"
        prec = f"{metrics.get('precision', 0) * 100:.1f}%"

        roc_auc = metrics.get('roc_auc', None)
        sensitivity = f"{metrics.get('sensitivity', 0) * 100:.1f}%"
        specificity = f"{metrics.get('specificity', 0) * 100:.1f}%"
        if isinstance(roc_auc, dict):
            # Average over classes
            auc_val = np.mean(list(roc_auc.values()))
        else:
            auc_val = roc_auc

        auc_str = f"{auc_val * 100:.1f}%" if auc_val is not None else "N/A"

        table_md += f"| {model_name} | {acc} | {auc_str} | {prec} | {sensitivity} | {specificity} |\n"

    return table_md


In [23]:
markdown_output = generate_markdown_table(cleaned_results)
print(markdown_output)

| Model | Accuracy | AUC (avg) | Precision | Sensitivity | Specificity
|--------|----------|------------|-----------|-----------|-----------|
| Frequency | 93.8% | 98.9% | 95.5% | 100.0% | 92.6% |
| Proximity | 94.7% | 86.0% | 95.0% | 66.7% | 100.0% |
| Proximity_1d | 92.0% | 88.3% | 92.7% | 50.0% | 100.0% |
| G-score | 90.3% | 90.8% | 91.7% | 83.3% | 91.6% |
| Frequency + Proximity | 98.2% | 99.4% | 98.3% | 88.9% | 100.0% |
| Frequency + G-score | 98.2% | 99.8% | 98.4% | 100.0% | 97.9% |
| Combined | 97.3% | 99.6% | 97.3% | 88.9% | 98.9% |

